# Combine pose datasets and fine-tune RTMPose-S

This notebook copies pose archives from Google Drive, clones the project, creates one cleaned COCO-17 database, checks sample overlays, runs a smoke test, and starts the full RTMPose-S fine-tune. Use a **T4 GPU** or better.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi

## Clone the Git repository

The existing folder is updated on a rerun instead of being deleted.

In [ ]:
from pathlib import Path

GIT_URL = 'https://github.com/abhicake2048/muay_thai_POC.git'
BRANCH = 'main'
REPO = '/content/Muay_thai'

if Path(REPO).is_dir():
    !git -C {REPO} pull --ff-only
else:
    !git clone --depth 1 --branch {BRANCH} {GIT_URL} {REPO}
%cd {REPO}

## Copy data from Drive to Colab

Before running this cell, upload the pose ZIP files to `MyDrive/Muay_thai/datasets/` using the Google Drive web interface. The important current archives are:

- `mma-fighter-pose-estimation-dataset.zip`
- `keypoint_mma_remove_no_point_data_coco.zip`
- `MMA_Pose_need_cleaning_coco.zip`

Detection-only archives are allowed; the merger reports and skips them.

In [ ]:
import shutil

drive_data = Path('/content/drive/MyDrive/Muay_thai/datasets')
local_data = Path(REPO) / 'datasets'
local_data.mkdir(parents=True, exist_ok=True)
archives = sorted(drive_data.glob('*.zip'))
assert archives, f'No ZIP archives found in {drive_data}'
for source in archives:
    destination = local_data / source.name
    print(f'Copying {source.name}: {source.stat().st_size / 1024**2:.1f} MiB')
    shutil.copy2(source, destination)
print(f'Copied {len(archives)} archive(s) to {local_data}')

## Install the pinned MMPose environment

In [ ]:
%cd {REPO}
!python -m pip install -q uv
!uv venv --python 3.11 --seed .venv-mmpose
!.venv-mmpose/bin/python install_mmpose_deps.py --torch cu121
!.venv-mmpose/bin/python -c "import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))"

## Merge, clean, deduplicate, and split

The complete database will be `annotations/all.json`; the other JSON files are leakage-aware train/validation/test views.

In [ ]:
DATA_ROOT = f'{REPO}/generated/combined_rtmpose_coco'

!.venv-mmpose/bin/python prepare_combined_pose_dataset.py \
  --input {REPO}/datasets \
  --output {DATA_ROOT} \
  --min-keypoints 4 \
  --train-ratio 0.80 \
  --val-ratio 0.10 \
  --render-samples 24 \
  --seed 42 \
  --overwrite

## Audit the result

Review the report and skeleton overlays. Stop and inspect the source mapping if the overlays are systematically wrong.

In [ ]:
import json
from IPython.display import display
from PIL import Image

root = Path(DATA_ROOT)
report = json.loads((root / 'dataset_report.json').read_text())
print(json.dumps({
    'combined': report['combined'],
    'splits': report['splits'],
    'cleaning': report['cleaning'],
    'group_overlaps': report['group_overlaps'],
}, indent=2))

for path in sorted((root / 'previews').glob('*'))[:12]:
    print(path.name)
    display(Image.open(path).resize((420, 420)))

## One-epoch smoke test

This intentionally uses only a small sample. It verifies CUDA, the MMPose data loader, the model, and COCO evaluation before the long run.

In [ ]:
SMOKE_DIR = '/content/drive/MyDrive/Muay_thai/runs/rtmpose_combined_smoke'

!.venv-mmpose/bin/python train_rtmpose.py \
  --data-root {DATA_ROOT} \
  --annotation-dir {DATA_ROOT}/annotations \
  --epochs 1 \
  --batch-size 32 \
  --workers 2 \
  --device cuda \
  --amp \
  --train-samples 256 \
  --validation-samples 64 \
  --work-dir {SMOKE_DIR}

## Full RTMPose-S fine-tune

Batch 64 is a safe starting point for a T4. Use 32 if CUDA runs out of memory, or try 128 on an L4/A100. Checkpoints are saved to Drive.

In [ ]:
RUN_DIR = '/content/drive/MyDrive/Muay_thai/runs/rtmpose_s_combined'

!.venv-mmpose/bin/python train_rtmpose.py \
  --data-root {DATA_ROOT} \
  --annotation-dir {DATA_ROOT}/annotations \
  --epochs 100 \
  --batch-size 64 \
  --workers 2 \
  --device cuda \
  --amp \
  --no-flip-test \
  --val-interval 5 \
  --checkpoint-interval 5 \
  --test-after-training \
  --work-dir {RUN_DIR}

## Find the best checkpoint or resume

The best checkpoint matches `best_coco_AP_epoch_*.pth`. To resume after a disconnect, rebuild the same dataset with seed 42, reinstall the environment, and add `--resume RESUME_FROM` to the full training command.

In [ ]:
run_dir = Path(RUN_DIR)
best = sorted(run_dir.glob('best_coco_AP*.pth'), key=lambda p: p.stat().st_mtime)
epochs = sorted(run_dir.glob('epoch_*.pth'), key=lambda p: p.stat().st_mtime)
print('Best:', best[-1] if best else 'not written yet')
print('Resume from:', epochs[-1] if epochs else 'no epoch checkpoint yet')